In [50]:
import astropy.io.fits as fits
import astropy.units as u
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
import aplpy
import pandas as pd
from scipy.ndimage import rotate
from astropy.wcs import WCS
from astropy.constants import c
from matplotlib.backends.backend_pdf import PdfPages
import os
import glob
import yaml

import warnings
warnings.filterwarnings("ignore")

In [51]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
IMAGE_DIRECTORY = config["data_dir"]
cont_files = glob.glob(f'{IMAGE_DIRECTORY}/*/*cont*.fits')
spw39_files = glob.glob(f'{IMAGE_DIRECTORY}/*/*12co*.fits') + glob.glob(f'{IMAGE_DIRECTORY}/*/*spw39*.fits')

In [52]:
data = []
for cont_file, spw39_file in zip(cont_files, spw39_files):
    cont = fits.open(cont_file)[0]
    spw39 = fits.open(spw39_file)[0]

    cont_bmaj = cont.header['BMAJ'] * 3600
    cont_bmin = cont.header['BMIN'] * 3600
    cont_rms = np.nanstd(cont.data[0,0,:,:])
    cont_length = cont.shape[2]
    cont_width = cont.shape[3]
    cont_xpixscale = np.abs(cont.header['CDELT1'] * 3600)
    cont_ypixscale = np.abs(cont.header['CDELT2'] * 3600)

    spw39_bmaj = spw39.header['BMAJ'] * 3600
    spw39_bmin = spw39.header['BMIN'] * 3600
    spw39_rms = np.nanstd(spw39.data[0,0,:,:])
    spw39_length = spw39.shape[2]
    spw39_width = spw39.shape[3]
    spw39_xpixscale = np.abs(spw39.header['CDELT1'] * 3600)
    spw39_ypixscale = np.abs(spw39.header['CDELT2'] * 3600)

    record = {'name': cont_file, 'cont_bmaj': cont_bmaj, 'cont_bmin': cont_bmin, 'cont_rms': cont_rms, 'cont_length': cont_length, 'cont_width': cont_width, 'cont_xpixscale': cont_xpixscale, 'cont_ypixscale': cont_ypixscale,
              'spw39_bmaj': spw39_bmaj, 'spw39_bmin': spw39_bmin, 'spw39_rms': spw39_rms, 'spw39_length': spw39_length, 'spw39_width': spw39_width, 'spw39_xpixscale': spw39_xpixscale, 'spw39_ypixscale': spw39_ypixscale}
    data.append(record)
df = pd.DataFrame(data)

In [53]:
df

,name,cont_bmaj,cont_bmin,cont_rms,cont_length,cont_width,cont_xpixscale,cont_ypixscale,spw39_bmaj,spw39_bmin,spw39_rms,spw39_length,spw39_width,spw39_xpixscale,spw39_ypixscale
0,/Volumes/Alpha/Research/data/hops-12/cont.fits,0.210106,0.146341,0.000133,1280,1280,0.035,0.035,0.218180,0.154732,0.002444,1372,1372,0.031,0.031
1,/Volumes/Alpha/Research/data/hops-138/cont.fits,0.179100,0.128759,0.000046,1280,1280,0.035,0.035,0.268478,0.179522,0.001719,1176,1176,0.036,0.036
2,/Volumes/Alpha/Research/data/hops-158/cont.fits,0.206553,0.144613,0.000038,1280,1280,0.035,0.035,0.212107,0.154954,0.002090,1372,1372,0.031,0.031
3,/Volumes/Alpha/Research/data/hops-163/cont.fits,0.194929,0.142247,0.000052,1280,1280,0.035,0.035,0.203847,0.151492,0.002446,1440,1440,0.030,0.030
4,/Volumes/Alpha/Research/data/hops-168/cont.fits,0.206340,0.147036,0.000217,1280,1280,0.035,0.035,0.186402,0.135598,0.002683,1620,1620,0.026,0.026
5,/Volumes/Alpha/Research/data/hops-173/cont.fits,0.203953,0.142428,0.000080,1280,1280,0.035,0.035,0.210531,0.152858,0.002473,1372,1372,0.031,0.031
6,/Volumes/Alpha/Research/data/hops-182/cont.fits,0.203743,0.145513,0.000703,1280,1280,0.035,0.035,0.186554,0.134891,0.003030,1620,1620,0.026,0.026
7,/Volumes/Alpha/Research/data/hops-193/cont.fits,0.197948,0.140993,0.000043,1280,1280,0.035,0.035,0.206461,0.150705,0.002336,1440,1440,0.030,0.030
8,/Volumes/Alpha/Research/data/hops-203/cont.fits,0.210765,0.148905,0.000164,1280,1280,0.035,0.035,0.218308,0.157367,0.002217,1372,1372,0.031,0.031
9,/Volumes/Alpha/Research/data/hops-213/cont.fits,0.246690,0.177168,0.000111,1280,1280,0.035,0.035,0.311179,0.247948,0.001334,1050,1050,0.041,0.041


In [54]:
df_orion = df[~df['name'].str.contains('per')]
df_perseus = df[df['name'].str.contains('per')]

In [55]:
df_orion.describe()

,cont_bmaj,cont_bmin,cont_rms,cont_length,cont_width,cont_xpixscale,cont_ypixscale,spw39_bmaj,spw39_bmin,spw39_rms,spw39_length,spw39_width,spw39_xpixscale,spw39_ypixscale
count,40.000000,40.000000,40.000000,40.0,40.0,40.000,40.000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000
mean,0.217565,0.166366,0.000198,1280.0,1280.0,0.035,0.035,0.227159,0.174544,0.002044,1277.450000,1277.450000,0.034450,0.034450
std,0.025666,0.037280,0.000281,0.0,0.0,0.000,0.000,0.027768,0.035195,0.000353,213.629436,213.629436,0.006484,0.006484
min,0.177651,0.128364,0.000037,1280.0,1280.0,0.035,0.035,0.186402,0.134891,0.001334,896.000000,896.000000,0.026000,0.026000
25%,0.200700,0.142087,0.000053,1280.0,1280.0,0.035,0.035,0.209027,0.150898,0.001716,1027.500000,1027.500000,0.030000,0.030000
50%,0.206357,0.145461,0.000080,1280.0,1280.0,0.035,0.035,0.213415,0.154272,0.002135,1372.000000,1372.000000,0.031000,0.031000
75%,0.247173,0.206673,0.000191,1280.0,1280.0,0.035,0.035,0.251892,0.221038,0.002220,1440.000000,1440.000000,0.041750,0.041750
max,0.262607,0.231222,0.001444,1280.0,1280.0,0.035,0.035,0.311179,0.247948,0.003030,1620.000000,1620.000000,0.047000,0.047000


In [58]:
df_perseus = df_perseus[~df_perseus['name'].str.contains('107')]
df_perseus.describe()

,cont_bmaj,cont_bmin,cont_rms,cont_length,cont_width,cont_xpixscale,cont_ypixscale,spw39_bmaj,spw39_bmin,spw39_rms,spw39_length,spw39_width,spw39_xpixscale,spw39_ypixscale
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.232250,0.159721,0.000504,1384.000000,1384.000000,0.032300,0.032300,0.291723,0.212294,0.003761,1014.400000,1014.400000,0.041500,0.041500
std,0.003694,0.005325,0.000524,44.621868,44.621868,0.001059,0.001059,0.004916,0.008724,0.000265,35.122642,35.122642,0.001434,0.001434
min,0.226144,0.151959,0.000123,1344.000000,1344.000000,0.031000,0.031000,0.283534,0.198937,0.003426,980.000000,980.000000,0.039000,0.039000
25%,0.230938,0.155251,0.000163,1344.000000,1344.000000,0.031250,0.031250,0.291118,0.205210,0.003541,985.000000,985.000000,0.040250,0.040250
50%,0.232779,0.161361,0.000423,1372.000000,1372.000000,0.032500,0.032500,0.292109,0.214651,0.003719,1000.000000,1000.000000,0.042000,0.042000
75%,0.234855,0.162880,0.000544,1430.000000,1430.000000,0.033000,0.033000,0.293593,0.218026,0.003981,1043.500000,1043.500000,0.042750,0.042750
max,0.237323,0.167531,0.001904,1440.000000,1440.000000,0.034000,0.034000,0.301244,0.223018,0.004190,1080.000000,1080.000000,0.043000,0.043000
